# Lesson 14: A real tokenizer: Byte-Pair Encoding (BPE)

**Why:** our model reads one character at a time, so "electricity" takes **11 steps**, and the 128-character context holds only about 20 words. Real GPTs read **pieces of words** ("elect", "ricity"), so each step carries more meaning and the same context holds much more text.

**The idea (BPE), like learning abbreviations:**
1. Start with the smallest units: the **256 possible bytes**. Every text in every language is made of them.
2. Find the **most common neighbouring pair** in the training text (say `t` + `h`).
3. Give that pair its own new token (`th`) and replace it everywhere.
4. Repeat. `th` + `e` → `the`, then ` ` + `the` → ` the`... until the vocabulary has the size you want.

This is how GPT-2, GPT-4 and Claude tokenize text. We'll build one from Tesla's own writing.

The code lives in [`src/tinygpt/bpe.py`](../src/tinygpt/bpe.py).

## 1. Bytes: the alphabet of every language

Computers store text as **bytes** (numbers 0–255) using a standard called **UTF-8**. Plain English letters take 1 byte each. Other letters take 2–4 bytes.

**Predict:** how many bytes is the Serbian letter **ć**? And the Cyrillic word **Тесла** (5 letters)?

In [ ]:
for word in ["Tesla", "ć", "Тесла", "⚡"]:
    b = word.encode("utf-8")
    print(f"{word!r:9} {len(word)} characters -> {len(b)} bytes: {list(b)}")

Because everything is bytes, a byte-based tokenizer can represent **any** text. Lesson 4's character tokenizer crashed on ć; this one never will.

## 2. One merge, by hand

Count every neighbouring pair of bytes in a small piece of Tesla-like text, and merge the winner.

In [ ]:
from collections import Counter
from tinygpt.bpe import merge

sample = "the theory of the thermal current in the coil"
ids = list(sample.encode("utf-8"))
pairs = Counter(zip(ids, ids[1:]))

print("most common pairs:")
for (a, b), n in pairs.most_common(5):
    print(f"   {chr(a)!r} + {chr(b)!r}  appears {n} times")

best = pairs.most_common(1)[0][0]
merged = merge(ids, best, 256)            # 256 = the first new token id
print(f"\nbefore: {len(ids)} tokens   after merging {chr(best[0])!r}+{chr(best[1])!r} -> 256: {len(merged)} tokens")

## 3. Splitting into word-like chunks first

If we merged freely, we'd get tokens like `"e. T"` that glue the end of one sentence to the start of the next. GPT-2 avoids that by first **splitting text into chunks** (words with their leading space, numbers, punctuation) and only merging **inside** a chunk.

In [ ]:
import regex
from tinygpt.bpe import GPT2_PATTERN

print(regex.findall(GPT2_PATTERN, "Tesla's motor ran at 1,500 revolutions, didn't it?"))

## 4. Train the tokenizer on Tesla's book

We learn **256 merges**, for a vocabulary of 256 + 256 = **512 tokens**. It takes about 15 seconds.

**Predict:** what will the **very first** merge be? (Remember: a word's leading space belongs to it, as in `" the"`.)

In [ ]:
import time
from tinygpt.bpe import BPETokenizer
from tinygpt.data import load_text

text = load_text()
start = time.time()
bpe = BPETokenizer.train(text, vocab_size=512)
print(f"learned {len(bpe.merges)} merges in {time.time() - start:.0f}s -> vocabulary of {bpe.vocab_size}")

learned = [bpe.vocab[i].decode("utf-8", errors="replace") for i in range(256, 512)]
print("\nfirst 20 merges:", learned[:20])
print("\nlast 20 merges: ", learned[-20:])

## 5. What Tesla's text looks like as tokens

The `|` marks show where each token starts and ends.

In [ ]:
sentence = "The alternating current motor produces a rotating magnetic field."
ids = bpe.encode(sentence)
print(f"{len(sentence)} characters -> {len(ids)} tokens")
print("|" + "|".join(bpe.token_strings(ids)) + "|")

all_ids = bpe.encode(text)
print(f"\nwhole book: {len(text):,} characters -> {len(all_ids):,} tokens  "
      f"({len(text) / len(all_ids):.2f} characters per token)")

## 6. Any language works now

In [ ]:
for s in ["Nikola Tesla, Smiljan: ćevapi i đak", "Никола Тесла", "⚡ 110 V"]:
    ids = bpe.encode(s)
    print(f"{s!r}: {len(ids)} tokens, round trip ok: {bpe.decode(ids) == s}")

## 7. Compared with real GPT tokenizers

`tiktoken` is OpenAI's tokenizer library. `gpt2` has 50,257 tokens learned from 40 GB of web text; `cl100k_base` (GPT-4) has about 100,000. (The first run downloads their vocabulary files.)

In [ ]:
import tiktoken

sentence = "The alternating current motor produces a rotating magnetic field."
rows = [("ours (Tesla, 512)", bpe.token_strings(bpe.encode(sentence)))]
for name in ["gpt2", "cl100k_base"]:
    enc = tiktoken.get_encoding(name)
    rows.append((f"{name} ({enc.n_vocab:,})", [enc.decode([t]) for t in enc.encode(sentence)]))

for name, toks in rows:
    print(f"{name:22} {len(toks):3d} tokens  |" + "|".join(toks) + "|")

## 8. Save it, and final check

We save the tokenizer so Lesson 15 can train a GPT with it.

In [ ]:
from tinygpt.data import PROJECT_ROOT

bpe.save(PROJECT_ROOT / "checkpoints" / "tesla_bpe_512.json")
assert bpe.decode(all_ids) == text
assert len(all_ids) < len(text) / 2
print("Saved checkpoints/tesla_bpe_512.json. Lesson 14 complete.")

## Wrap-up question

A bigger vocabulary means fewer tokens per text. So why don't we just use 100,000 tokens like GPT-4, for our small model and small Tesla dataset? What would go wrong?